In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
shashanknecrothapa_ames_housing_dataset_path = kagglehub.dataset_download('shashanknecrothapa/ames-housing-dataset')

print('Data source import complete.')


Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv


# 🤖 Autonomous Data Science Agent — Interactive Evaluation Sandbox

Welcome Judges! This notebook provides a fully self-contained, live runtime environment for the Autonomous Data Science Agent. It automatically checks data integrity, sandboxes dynamic code generation, and executes real-time pipeline iterations using **Google Gemini 2.5 Flash**.

### 🚀 Quick-Start Instructions for Evaluation:
1. **Add Your Gemini API Key:** On the right-hand panel of this notebook, expand **Add-ons** ➡️ **Secrets**. Add a secret with the Label Name exactly as `GEMINI_API_KEY` and paste your valid Google AI Studio API key. Ensure the toggle switch for notebook access is turned **ON**.
2. **Set Your Target Variable:** If you wish to test a completely different prediction goal, simply update the `CHOSEN_TARGET_COLUMN` variable string in the execution cell below to match any column inside the dataset.
3. **Execute:** Click **Run All** in the notebook toolbar. Watch the agent loop dynamically construct state updates, handle feature expansions, and track convergence completely unassisted!

In [3]:
import os
import glob

# FORCE KAGGLE TO SYNC THE NEW DISK-PERSISTENCE FIX FROM GITHUB
print("🔄 Fetching the latest patched code from GitHub...")
%cd /kaggle/working

repo_dir = "/kaggle/working/autonomous-ds-agent"
repo_url = "https://github.com/25f3000007-code/autonomous-ds-agent.git"

if os.path.exists(repo_dir):
    print("📂 Repository found locally. Syncing latest changes...")
    %cd {repo_dir}
    !git reset --hard HEAD
    !git pull origin main
else:
    print("📥 Repository not found. Initializing fresh clone from GitHub...")
    !git clone {repo_url}
    %cd {repo_dir}

print("✅ Workspace successfully synchronized with your latest GitHub commit!")
print("-" * 60)


# List files to see the exact name of your cloned folder
print(os.listdir('/kaggle/working/'))

# Change directory into your repository (replace with your exact folder name)
os.chdir('/kaggle/working/autonomous-ds-agent')
print("Current Working Directory:", os.getcwd())

🔄 Fetching the latest patched code from GitHub...
/kaggle/working
📥 Repository not found. Initializing fresh clone from GitHub...
Cloning into 'autonomous-ds-agent'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 183 (delta 114), reused 143 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.46 MiB | 11.71 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/kaggle/working/autonomous-ds-agent
✅ Workspace successfully synchronized with your latest GitHub commit!
------------------------------------------------------------
['autonomous-ds-agent', '__notebook__.ipynb']
Current Working Directory: /kaggle/working/autonomous-ds-agent


In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 96.3 MB/s eta 0:00:00


In [5]:
import time
import uuid
import sys
import subprocess
import pandas as pd
import os
import glob
from kaggle_secrets import UserSecretsClient

# 🎯 STEP 1: CHOOSE YOUR TARGET COLUMN
CHOSEN_TARGET_COLUMN = "SalePrice"

print("🚀 INITIATING AUTOMATED ARTIFACT PIPELINE")
print("=" * 60)

# 🔄 STEP 2: FORCE KAGGLE TO SYNC THE NEW DISK-PERSISTENCE FIX FROM GITHUB
print("🔄 Fetching the latest patched code from GitHub...")
%cd /kaggle/working

repo_dir = "/kaggle/working/autonomous-ds-agent"
repo_url = "https://github.com/25f3000007-code/autonomous-ds-agent.git"

if os.path.exists(repo_dir):
    print("📂 Repository found locally. Syncing latest changes...")
    %cd {repo_dir}
    !git reset --hard HEAD
    !git pull origin main
else:
    print("📥 Repository not found. Initializing fresh clone from GitHub...")
    !git clone {repo_url}
    %cd {repo_dir}

print("✅ Workspace successfully synchronized with your latest GitHub commit!")
print("-" * 60)

# 🔑 STEP 3: SETUP ENVIRONMENT KEYS
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
# Optimization: Switching to gemini-1.5-flash for better stability during congestion
os.environ["MODEL_NAME"] = "gemini-1.5-flash"

# 📂 STEP 4: LOCATE DATASET
kaggle_input_path = "/kaggle/input/datasets/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv"
if not os.path.exists(kaggle_input_path):
    input_csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    if input_csv_files: kaggle_input_path = input_csv_files[0]

print(f"📂 Found input dataset at: {kaggle_input_path}")

# 🗺️ STEP 5: PREPARE DATA
df = pd.read_csv(kaggle_input_path)
if CHOSEN_TARGET_COLUMN in df.columns:
    if 'price' in df.columns and CHOSEN_TARGET_COLUMN != 'price':
        df.rename(columns={'price': 'original_price_col'}, inplace=True)
    df.rename(columns={CHOSEN_TARGET_COLUMN: 'price'}, inplace=True)

    os.makedirs('data', exist_ok=True)
    local_target_path = 'data/uploaded_dataset.csv'
    df.to_csv(local_target_path, index=False)

    # 🤖 STEP 6: KICK OFF AGENT
    print("🚀 Booting up Autonomous Data Science Agent Loop...")
    log_file_path = "/kaggle/working/trial_execution.log"
    cmd = "PYTHONPATH=. python src/main.py --dataset data/uploaded_dataset.csv --target price"

    with open(log_file_path, "w") as log_file:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            sys.stdout.write(line)
            log_file.write(line)
            log_file.flush()
    process.wait()

    # 💾 STEP 7: SAVE FINAL SUBMISSION
    if os.path.exists(local_target_path):
        df_final = pd.read_csv(local_target_path)
        df_final.rename(columns={'price': CHOSEN_TARGET_COLUMN}, inplace=True)
        if 'original_price_col' in df_final.columns:
            df_final.rename(columns={'original_price_col': 'price'}, inplace=True)

        df_final.to_csv('/kaggle/working/submission.csv', index=False)
        print("\n🎯 FINAL OPTIMIZED SUBMISSION SAVED: /kaggle/working/submission.csv")
else:
    print(f"❌ Target '{CHOSEN_TARGET_COLUMN}' not found.")

🚀 INITIATING AUTOMATED ARTIFACT PIPELINE
🔄 Fetching the latest patched code from GitHub...
/kaggle/working
📂 Repository found locally. Syncing latest changes...
/kaggle/working/autonomous-ds-agent
HEAD is now at 9c10a08 Improve output artifact generation on data processing failures
From https://github.com/25f3000007-code/autonomous-ds-agent
 * branch            main       -> FETCH_HEAD
Already up to date.
✅ Workspace successfully synchronized with your latest GitHub commit!
------------------------------------------------------------
📂 Found input dataset at: /kaggle/input/datasets/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv
🚀 Booting up Autonomous Data Science Agent Loop...

STARTING AUTONOMOUS DATA SCIENCE AGENT
📋 Run ID: 66DF62 | Timestamp: 20260623_115941
✅ [Monitor] Successfully loaded dataset: /kaggle/working/autonomous-ds-agent/data/uploaded_dataset.csv
📊 [Agent] Baseline RMSE (Lower is better): 24812.3771
⏳ Giving Gemini API a brief cooling period...
🧠 [Brain] Reque